In [0]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  NOTEBOOK 01 — BRONZE INGESTION                                         ║
# ║                                                                         ║
# ║  PIPELINE FLOW (from architecture diagram):                             ║
# ║                                                                         ║
# ║  Raw CSV arrives                                                        ║
# ║       │                                                                 ║
# ║       ▼                                                                 ║
# ║  1 — SECURITY                                                           ║
# ║       dbutils.secrets.get() — never hardcode credentials                ║
# ║       │                                                                 ║
# ║       ▼                                                                 ║
# ║  2 — CDC (Change Data Capture)                                          ║
# ║       get_new_files() — skip already-processed files                    ║
# ║       │                          │                                      ║
# ║       │ (no new files)      (new files found)                          ║
# ║       ▼                          │                                      ║
# ║      SKIP                        ▼                                      ║
# ║                           3 — SCHEMA EVOLUTION                         ║
# ║                                detect_and_evolve_schema()               ║
# ║                                compare columns                          ║
# ║                           │              │             │                ║
# ║                     col added       no change     col removed           ║
# ║                           │              │             │                ║
# ║                  COLUMN_ADDED      Continue    COLUMN_REMOVED           ║
# ║                  schema_evolved          │    NULL-fill, warn           ║
# ║                  =True, mergeSchema      │    pipeline CONTINUES        ║
# ║                           │              │             │                ║
# ║                           └──────────────┴─────────────┘               ║
# ║                                          │                              ║
# ║                                          ▼                              ║
# ║                                4 — SCHEMA VERSIONING                   ║
# ║                                   write to schema_versions table        ║
# ║                                   acknowledged=false                   ║
# ║                                          │                              ║
# ║                                          ▼                              ║
# ║                                 Write to Delta (APPEND)                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import logging
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType, BooleanType
)

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — all paths hardcoded, no external imports needed
# ══════════════════════════════════════════════════════════════════════════════

CATALOG  = "retail_data_project"
BRONZE   = f"{CATALOG}.01_bronze"
METADATA = f"{CATALOG}.04_metadata"
RAW_BASE = f"/Volumes/{CATALOG}/01_bronze/raw_data"

CUSTOMER_TABLE  = f"{BRONZE}.customers"
PRODUCT_TABLE   = f"{BRONZE}.products"
SALES_TABLE     = f"{BRONZE}.sales"
META_TABLE      = f"{METADATA}.processed_files"
PIPELINE_RUNS   = f"{METADATA}.pipeline_runs"
SCHEMA_VERSIONS = f"{METADATA}.schema_versions"


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — SECURITY
# ──────────────────────────────────────────────────────────────────────────
# All credentials retrieved from Databricks secret scope.
# NEVER hardcode passwords or tokens in notebooks.
#
# Setup (run once in terminal):
#   databricks secrets create-scope retail-pipeline
#   databricks secrets put-secret retail-pipeline snowflake-password
#   databricks secrets put-secret retail-pipeline snowflake-user
#   databricks secrets put-secret retail-pipeline snowflake-account
#
# Community Edition: secret scopes are not available.
# The fallback="" means the pipeline still runs without them.
# Snowflake load (Notebook 04) will skip if credentials are empty.
# ══════════════════════════════════════════════════════════════════════════════

def get_secret(scope: str, key: str, fallback: str = "") -> str:
    """
    Retrieves a secret from Databricks secret scope.
    Falls back to fallback value if scope is unavailable (Community Edition).
    Returned value is automatically REDACTED in Databricks notebook output.
    """
    try:
        return dbutils.secrets.get(scope=scope, key=key)
    except Exception:
        return fallback

SNOWFLAKE_USER     = get_secret("retail-pipeline", "snowflake-user")
SNOWFLAKE_PASSWORD = get_secret("retail-pipeline", "snowflake-password")
SNOWFLAKE_ACCOUNT  = get_secret("retail-pipeline", "snowflake-account")


# ══════════════════════════════════════════════════════════════════════════════
# STRUCTURED LOGGER
# ══════════════════════════════════════════════════════════════════════════════

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            "%Y-%m-%d %H:%M:%S"
        ))
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger

logger    = get_logger("01_bronze_ingestion")
RUN_START = datetime.now()

try:
    RUN_DATE = dbutils.widgets.get("run_date")
except Exception:
    RUN_DATE = str(datetime.now().date())

logger.info(f"Bronze ingestion STARTED | run_date={RUN_DATE}")
logger.info(f"Security: secrets loaded via dbutils.secrets.get()")


# ══════════════════════════════════════════════════════════════════════════════
# EXPECTED SCHEMAS — all StringType
# Bronze is a raw landing zone. Type casting happens in Silver (Notebook 02).
# nullable=True so PERMISSIVE mode keeps rows with missing values as NULL.
# ══════════════════════════════════════════════════════════════════════════════

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("name",        StringType(), True),
    StructField("email",       StringType(), True),   # PII — masked in silver
    StructField("city",        StringType(), True),
    StructField("signup_date", StringType(), True),
])

product_schema = StructType([
    StructField("product_id",   StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category",     StringType(), True),
    StructField("price",        StringType(), True),  # may be NULL/empty
])

sales_schema = StructType([
    StructField("order_id",    StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id",  StringType(), True),
    StructField("quantity",    StringType(), True),
    StructField("order_date",  StringType(), True),
])


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — CDC (CHANGE DATA CAPTURE)
# ──────────────────────────────────────────────────────────────────────────
# processed_files is our CDC state store.
# On every run:
#   all_files  = every CSV currently in the Volume folder
#   processed  = files already recorded in processed_files table
#   new_files  = all_files - processed  ← only these get ingested
#
# Run 1: processed=[]            → ingest all files
# Run 2: processed=[file1.csv]   → ingest only file2.csv, file3.csv ...
#
# WHY file-level CDC and not row-level?
#   File-level CDC skips the read entirely for already-seen files.
#   Row deduplication still happens in silver via MERGE on order_id.
# ══════════════════════════════════════════════════════════════════════════════

def get_new_files(folder_path: str, table_name: str) -> list:
    """
    CDC: returns only CSV files in folder_path not yet in processed_files.
    Returns [] → pipeline SKIPs (no read, no write, no state update).
    Returns [paths] → pipeline continues to schema evolution.
    """
    try:
        all_files = [
            f.path for f in dbutils.fs.ls(folder_path)
            if f.name.endswith(".csv")
        ]
    except Exception as e:
        logger.error(f"[CDC][{table_name}] Cannot list '{folder_path}': {e}")
        return []

    if not all_files:
        logger.info(f"[CDC][{table_name}] No CSV files in {folder_path} → SKIP")
        return []

    try:
        processed = [
            row.file_name for row in
            spark.table(META_TABLE)
                .filter(F.col("table_name") == table_name)
                .select("file_name")
                .collect()
        ]
    except Exception:
        processed = []   # first run — table empty, all files are new

    new_files = list(set(all_files) - set(processed))

    logger.info(
        f"[CDC][{table_name}] "
        f"Found={len(all_files)} | Processed={len(processed)} | New={len(new_files)}"
    )

    if not new_files:
        logger.info(f"[CDC][{table_name}] All files already processed → SKIP")

    return new_files


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — SCHEMA EVOLUTION
# ──────────────────────────────────────────────────────────────────────────
# Three outcomes from the diagram:
#
#   COLUMN_ADDED   (col in CSV, not in expected schema)
#     → Kept as-is in bronze (capture everything at landing)
#     → mergeSchema=True on write allows it into Delta table
#     → schema_evolved=True, action="KEPT_EXTRA"
#
#   NO CHANGE  (CSV exactly matches expected schema)
#     → Continue normally, schema_evolved=False
#
#   COLUMN_REMOVED  (expected col missing from CSV)
#     → NULL-FILL: column added with NULL values    ← KEY DECISION
#     → Pipeline CONTINUES — does NOT halt
#     → schema_drift_flag=True on all rows from this file
#     → Logged to schema_versions with acknowledged=False
#     → Silver handles the NULL (drop or impute)
#     → action="NULL_FILLED"
#
# WHY NULL-FILL instead of halting?
#   Halting blocks ALL data — even complete rows from other files.
#   NULL-filling keeps data flowing while making the problem visible.
#   Silver sees schema_drift_flag=True and can apply per-column logic.
# ══════════════════════════════════════════════════════════════════════════════

def detect_and_evolve_schema(df, expected_schema, table_name: str, source_file: str):
    """
    Detects schema drift. Applies evolution (null-fill / keep extra).
    Pipeline NEVER halts here.

    Returns:
        (evolved_df, schema_drifted: bool, missing_cols: set, extra_cols: set)
    """
    expected_cols = {f.name for f in expected_schema.fields}
    actual_cols   = set(df.columns)
    missing_cols  = expected_cols - actual_cols   # COLUMN_REMOVED
    extra_cols    = actual_cols   - expected_cols  # COLUMN_ADDED
    schema_drifted = bool(missing_cols or extra_cols)

    # COLUMN_REMOVED: null-fill missing columns
    if missing_cols:
        logger.warning(
            f"[SCHEMA_EVOLUTION][{table_name}] COLUMN_REMOVED: {missing_cols} | "
            f"Action: NULL_FILLED | Pipeline continues | "
            f"Rows flagged with schema_drift_flag=True"
        )
        for col_name in missing_cols:
            df = df.withColumn(col_name, F.lit(None).cast(StringType()))

    # COLUMN_ADDED: keep in bronze, mergeSchema handles Delta write
    if extra_cols:
        logger.warning(
            f"[SCHEMA_EVOLUTION][{table_name}] COLUMN_ADDED: {extra_cols} | "
            f"Action: KEPT_EXTRA | mergeSchema=True on write"
        )

    if not schema_drifted:
        logger.info(f"[SCHEMA_EVOLUTION][{table_name}] NO_CHANGE — schema matches expected")

    return df, schema_drifted, missing_cols, extra_cols


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — SCHEMA VERSIONING
# ──────────────────────────────────────────────────────────────────────────
# Every drift event is logged to schema_versions with acknowledged=False.
# Data stewards review unacknowledged events:
#   SELECT * FROM retail_data_project.04_metadata.schema_versions
#   WHERE acknowledged = false
# After review: UPDATE ... SET acknowledged = true
# ══════════════════════════════════════════════════════════════════════════════

def record_schema_version(
    table_name: str,
    missing_cols: set,
    extra_cols: set,
    source_file: str,
    action_taken: str
) -> None:
    """Writes one audit row to schema_versions for each drift event."""
    try:
        schema = StructType([
            StructField("detected_at",     TimestampType(), False),
            StructField("table_name",      StringType(),    False),
            StructField("missing_columns", StringType(),    True),
            StructField("extra_columns",   StringType(),    True),
            StructField("source_file",     StringType(),    True),
            StructField("action_taken",    StringType(),    False),
            StructField("acknowledged",    BooleanType(),   False),
        ])
        row = spark.createDataFrame(
            [(
                datetime.now(),
                table_name,
                ", ".join(sorted(missing_cols)) if missing_cols else None,
                ", ".join(sorted(extra_cols))   if extra_cols   else None,
                source_file,
                action_taken,
                False   # acknowledged=False — requires data steward review
            )],
            schema=schema
        )
        row.write.format("delta").mode("append").saveAsTable(SCHEMA_VERSIONS)
        logger.info(
            f"[SCHEMA_VERSIONING][{table_name}] "
            f"Drift event recorded | acknowledged=False"
        )
    except Exception as e:
        logger.warning(f"[SCHEMA_VERSIONING] Could not write: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# OBSERVABILITY HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def null_profile(df, table_name: str) -> dict:
    """
    Logs null/empty value counts per column.
    WARNING level if any column exceeds 10% nulls.
    Returns {col: count} dict for programmatic use.
    """
    total   = df.count()
    profile = {}
    for col_name in df.columns:
        n = df.filter(
            F.col(col_name).isNull() |
            (F.col(col_name) == "") |
            (F.col(col_name) == "NULL")
        ).count()
        if n > 0:
            pct = round(n / total * 100, 1) if total > 0 else 0
            profile[col_name] = n
            level = logging.WARNING if pct > 10 else logging.INFO
            logger.log(level, f"  [NULL_PROFILE][{table_name}] '{col_name}': {n:,} nulls ({pct}%)")
    if not profile:
        logger.info(f"  [NULL_PROFILE][{table_name}] No nulls detected ✓")
    return profile


def check_row_count_anomaly(table_name: str, current_count: int, threshold: float = 0.50) -> None:
    """
    Warns if today's row count deviates >50% from historical average.
    Catches upstream truncations early before they reach dashboards.
    """
    try:
        hist_avg = (
            spark.table(META_TABLE)
            .filter(F.col("table_name") == table_name)
            .agg(F.avg("row_count").alias("avg"))
            .collect()[0]["avg"]
        )
        if hist_avg and hist_avg > 0:
            deviation = abs(current_count - hist_avg) / hist_avg
            if deviation > threshold:
                logger.warning(
                    f"[ROW_ANOMALY][{table_name}] "
                    f"Current={current_count:,} | Hist avg={hist_avg:,.0f} | "
                    f"Deviation={deviation:.1%} > {threshold:.0%} threshold → "
                    f"possible upstream truncation!"
                )
            else:
                logger.info(
                    f"[ROW_ANOMALY][{table_name}] Count normal: {current_count:,} "
                    f"(hist avg: {hist_avg:,.0f})"
                )
    except Exception:
        logger.info(f"[ROW_ANOMALY][{table_name}] First run — no baseline yet")


def record_pipeline_run(run_id, notebook, run_date, start_ts, status, rows, message):
    """Writes one row to pipeline_runs for SLA monitoring."""
    try:
        schema = StructType([
            StructField("run_id",         StringType(),    False),
            StructField("notebook_name",  StringType(),    False),
            StructField("run_date",       StringType(),    False),
            StructField("start_ts",       TimestampType(), False),
            StructField("end_ts",         TimestampType(), False),
            StructField("status",         StringType(),    False),
            StructField("rows_processed", LongType(),      False),
            StructField("message",        StringType(),    True),
        ])
        spark.createDataFrame(
            [(run_id, notebook, run_date, start_ts,
              datetime.now(), status, int(rows), str(message)[:500])],
            schema=schema
        ).write.format("delta").mode("append").saveAsTable(PIPELINE_RUNS)
    except Exception as e:
        logger.warning(f"[PIPELINE_RUNS] Could not write: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN INGESTION FUNCTION — IMPLEMENTS THE FULL DIAGRAM FLOW
# ══════════════════════════════════════════════════════════════════════════════

def ingest_entity(
    folder_path: str,
    expected_schema: StructType,
    table_name: str,
    delta_table: str
) -> int:
    """
    Executes the complete bronze ingestion flow for one entity.
    Flow: CDC → [SKIP] → Schema Evolution → Schema Versioning → Write Delta

    Returns: int rows ingested (0 = skipped, no new files)
    """
    logger.info(f"\n{'═' * 60}")
    logger.info(f"  ENTITY: {table_name.upper()}")
    logger.info(f"{'═' * 60}")

    # STEP 2: CDC — find only new files
    new_files = get_new_files(folder_path, table_name)
    if not new_files:
        logger.info(f"  [{table_name}] CDC → SKIP")
        return 0

    logger.info(f"  [{table_name}] CDC → {len(new_files)} new file(s) to process")

    # Read CSV without schema enforcement — schema evolution will align
    # PERMISSIVE: keeps rows even with parse errors (bad fields → null)
    # emptyValue=None: empty strings → null for consistent null handling
    # inferSchema=False: stay as StringType — silver does all casting
    df = (
        spark.read
        .option("header",      True)
        .option("mode",        "PERMISSIVE")
        .option("emptyValue",  None)
        .option("inferSchema", False)
        .csv(new_files)
    )
    logger.info(f"  [{table_name}] Read {df.count():,} rows")

    # STEP 3: SCHEMA EVOLUTION
    source_repr = new_files[0] if len(new_files) == 1 else f"{len(new_files)} files"
    df, schema_drifted, missing_cols, extra_cols = detect_and_evolve_schema(
        df, expected_schema, table_name, source_repr
    )

    # Null profile (observability only — not a rejection gate)
    null_profile(df, table_name)

    # STEP 4: SCHEMA VERSIONING — log any drift events
    if schema_drifted:
        actions = []
        if missing_cols: actions.append("NULL_FILLED")
        if extra_cols:   actions.append("KEPT_EXTRA")
        record_schema_version(
            table_name   = table_name,
            missing_cols = missing_cols,
            extra_cols   = extra_cols,
            source_file  = source_repr,
            action_taken = " + ".join(actions)
        )

    # Add pipeline lineage columns
    df = (
        df
        .withColumn("received_date",     F.current_date())
        .withColumn("source_file",       F.col("_metadata.file_path"))
        .withColumn("ingestion_ts",      F.current_timestamp())
        .withColumn("schema_drift_flag", F.lit(schema_drifted))
    )

    row_count = df.count()

    # Row count anomaly check
    check_row_count_anomaly(table_name, row_count)

    # WRITE TO DELTA — always APPEND (bronze is immutable)
    # mergeSchema=True: allows COLUMN_ADDED extra columns into Delta safely
    # Missing columns were already null-filled above — no schema conflict
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(delta_table)
    )
    logger.info(
        f"  [{table_name}] ✓ Appended {row_count:,} rows → {delta_table} "
        f"| schema_drift_flag={schema_drifted}"
    )

    # Update CDC state — record processed files
    # These paths appear in "already processed" on the next run → SKIP
    cdc_schema = StructType([
        StructField("file_name",           StringType(),    False),
        StructField("table_name",          StringType(),    False),
        StructField("processed_timestamp", TimestampType(), False),
        StructField("row_count",           LongType(),      False),
    ])
    cdc_rows = spark.createDataFrame(
        [(f, table_name, datetime.now(), int(row_count)) for f in new_files],
        schema=cdc_schema
    )
    cdc_rows.write.format("delta").mode("append").saveAsTable(META_TABLE)
    logger.info(
        f"  [{table_name}] CDC state updated — "
        f"{len(new_files)} file(s) recorded in processed_files"
    )

    return row_count


# ══════════════════════════════════════════════════════════════════════════════
# EXECUTION ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

RUN_ID = f"bronze_{RUN_DATE}_{datetime.now().strftime('%H%M%S')}"

try:
    record_pipeline_run(RUN_ID, "01_bronze", RUN_DATE,
                        RUN_START, "RUNNING", 0, "Started")

    total_rows = 0

    total_rows += ingest_entity(
        folder_path     = f"{RAW_BASE}/customers/",
        expected_schema = customer_schema,
        table_name      = "customers",
        delta_table     = CUSTOMER_TABLE
    )
    total_rows += ingest_entity(
        folder_path     = f"{RAW_BASE}/products/",
        expected_schema = product_schema,
        table_name      = "products",
        delta_table     = PRODUCT_TABLE
    )
    total_rows += ingest_entity(
        folder_path     = f"{RAW_BASE}/sales/",
        expected_schema = sales_schema,
        table_name      = "sales",
        delta_table     = SALES_TABLE
    )

    duration = (datetime.now() - RUN_START).seconds
    record_pipeline_run(RUN_ID, "01_bronze", RUN_DATE, RUN_START,
                        "SUCCESS", total_rows,
                        f"Ingested {total_rows:,} rows in {duration}s")

    logger.info(f"\n{'═'*60}")
    logger.info(f"  BRONZE INGESTION COMPLETE")
    logger.info(f"  Total rows : {total_rows:,}")
    logger.info(f"  Duration   : {duration}s")
    logger.info(f"{'═'*60}")

    # Bronze table totals with drift summary
    for tbl, name in [
        (CUSTOMER_TABLE, "customers"),
        (PRODUCT_TABLE,  "products"),
        (SALES_TABLE,    "sales"),
    ]:
        total   = spark.table(tbl).count()
        drifted = spark.table(tbl).filter(F.col("schema_drift_flag") == True).count()
        logger.info(
            f"  {name:12s} → total: {total:>10,} | drift-flagged: {drifted:,}"
        )

    # Alert on any unacknowledged schema drift
    try:
        unack = spark.table(SCHEMA_VERSIONS).filter(
            F.col("acknowledged") == False
        ).count()
        if unack > 0:
            logger.warning(
                f"\n  ⚠ {unack} unacknowledged schema drift event(s) need review."
                f"\n  → SELECT * FROM {SCHEMA_VERSIONS} WHERE acknowledged=false"
            )
    except Exception:
        pass

except Exception as e:
    record_pipeline_run(RUN_ID, "01_bronze", RUN_DATE, RUN_START,
                        "FAILED", 0, str(e)[:500])
    logger.error(f"Bronze ingestion FAILED: {e}")
    raise   # re-raise so Airflow marks task as FAILED

In [0]:
%sql
select * from retail_data_project.04_metadata.processed_files

In [0]:
%sql
select count(*) from retail_data_project.01_bronze.customers;

In [0]:
%sql
select count(*) from retail_data_project.01_bronze.products;

In [0]:
%sql
select count(*) from retail_data_project.01_bronze.sales

In [0]:
%sql
select * from retail_data_project.04_metadata.schema_versions;